In [1]:
import torch
import numpy as np
import drone_2d_custom_gym_env.drone_2d_env
if not hasattr(np, 'bool8'):
    np.bool8 = np.bool_
import gym
from ppo import PPO as CustomPPO
from utils import plot_learning_curve, Logger
from stable_baselines3 import PPO, SAC
from stable_baselines3.common.vec_env import DummyVecEnv

pygame-ce 2.5.7 (SDL 2.32.10, Python 3.12.12)


objc[80322]: Class SDLApplication is implemented in both /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x149ba4a68) and /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x151dd4890). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[80322]: Class SDLAppDelegate is implemented in both /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x149ba4ab8) and /Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x151dd48e0). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[80322]: Class SDLTranslatorResponder is implemented

In [2]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
CASE_ID = 1

CASES = {
    1: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force" : 600,"wind":None,"wind_magnitude":100},
    2: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind":None,"wind_magnitude":100},
    3: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force": 600,"wind": "Uniform","wind_magnitude":100},
    4: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind": "Uniform","wind_magnitude":100},
    5: {"initial_throw": True, "initial_force": 5000, "initial_rotation_force": 600,"wind": "Random","wind_magnitude":100},
    6: {"initial_throw": True, "initial_force": 12000, "initial_rotation_force": 1500,"wind": "Random","wind_magnitude":100}
}

def make_env(case_id, render_sim = False):
    case = CASES[case_id]
    return gym.make(
        "drone-2d-custom-v0",
        render_sim=render_sim,
        render_path=True,
        render_shade=True,
        shade_distance = 75,
        n_steps = 500,
        n_fall_steps = 5,
        change_target=False,
        initial_throw=case["initial_throw"],
        initial_force=case["initial_force"],
        initial_rotation_force=case["initial_rotation_force"],
        step_penalty = 0,
        goal_reward = 20.0,
        death_penalty = -10.0,
        success_radius = 25.0
    )

NB_EPISODES = 750
EVAL_EP_FREQ = 50

# training_env = DummyVecEnv([lambda: make_env(CASE_ID)])
# eval_env = DummyVecEnv([lambda: make_env(CASE_ID)])
training_env = make_env(CASE_ID)
eval_env = make_env(CASE_ID)

In [2]:
ppo_agent = CustomPPO(
    training_env = training_env,
    eval_env = eval_env,
    policy_neurons_size = [256, 256],
    value_neurons_size = [256, 256],
    lr_policy = 1e-4,
    lr_value = 1e-4,
    gamma = 0.99,
    lamda = 0.95,
    time_steps_per_batch = 2048,
    epochs = 10,
    epsilon = 0.2,
    mini_batch_size = 64,
    nb_eval_episodes = 200,
    logger = Logger.PPO(CASE_ID),
)

eval_mean_returns = []
eval_std_returns = []
current_best = -np.inf

for ep in range(NB_EPISODES):
    
    ppo_agent.learn()
    
    if ep % EVAL_EP_FREQ == 0:
        avg_return, std_return = ppo_agent.evaluation_phase()
        eval_mean_returns.append(avg_return)
        eval_std_returns.append(std_return)

        if avg_return > current_best:
            current_best = avg_return

        print(f'ep: {ep}, avg eval return {round(avg_return, 2)}, best avg eval return: {round(current_best, 2)}')

ppo_agent.logger.log_training_metrics()

NameError: name 'training_env' is not defined

In [ ]:
TIMESTEPS = 180000


model = PPO('MlpPolicy', training_env, verbose = 1, ent_coef = 0.01)
model.learn(total_timesteps = TIMESTEPS)
# if __name__ == "__main__":
#     env = make_env(CASE_ID)
#     try:
#         if ALGO == "ppo":
#             model = PPO("MlpPolicy", env, verbose=1)
#         elif ALGO == "sac":
#             model = SAC("MlpPolicy", env, verbose=1)
#         else:
#             raise ValueError(f"Unsupported algorithm: {ALGO}")

#         model.learn(total_timesteps=TIMESTEPS)
#         model.save(str(MODEL_DIR / f"{ALGO}_agent_case{CASE_ID}"))
#     finally:
#         env.close()

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(
/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/Users/oli.dmrs/Documents/Personal Projects/COMP579_FinalProject/.venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initial

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 87.2     |
|    ep_rew_mean     | -15.6    |
| time/              |          |
|    fps             | 8183     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 82.9         |
|    ep_rew_mean          | -15          |
| time/                   |              |
|    fps                  | 5416         |
|    iterations           | 2            |
|    time_elapsed         | 0            |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0054757074 |
|    clip_fraction        | 0.0312       |
|    clip_range           | 0.2          |
|    entropy_loss         | -2.86        |
|    explained_variance   | -0.0181      |
|    learning_r

In [ ]:
plot_learning_curve(
    x = np.arange(1, len(eval_mean_returns) + 1),
    y = np.array(eval_mean_returns),
    title = 'PPO evaluation phase average return',
    xlabel = 'Evaluation phase',
    ylabel = 'Return',
    std_values = np.array(eval_std_returns),
    legend_labels = ['PPO']
)